# Análisis Exploratorio de Datos (EDA) - Rappi
Esta notebook contiene un análisis exploratorio básico de los datos recolectados de Rappi México.

In [66]:
import pandas as pd
import json
import plotly.express as px
import plotly.io as pio

# Plantilla de Plotly
pio.templates.default = "plotly_white"

## 1. Carga de Datos

In [67]:
with open('../data/raw/rappi_products.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.head()

,timestamp,app_name,store_name,full_address,lat,lng,city,municipality,target_product,scraped_product_name,original_price,final_price,discount_amount,delivery_fee,eta,html_snippet
0,2026-04-18 19:20:35.110536,Rappi,McDonald's,"Polanco, Miguel Hidalgo, Ciudad de México",19.431252,-99.196062,Ciudad de México,Miguel Hidalgo,Coca-Cola mediana,Coca-Cola mediana,55.0,55.0,0.0,0.0,12,"<div class=""chakra-stack css-46p1lt"" data-qa=""..."
1,2026-04-18 19:20:35.110536,Rappi,McDonald's,"Polanco, Miguel Hidalgo, Ciudad de México",19.431252,-99.196062,Ciudad de México,Miguel Hidalgo,Big Mac,Big Mac,125.0,125.0,0.0,0.0,12,"<div class=""chakra-stack css-46p1lt"" data-qa=""..."
2,2026-04-18 19:20:35.110536,Rappi,McDonald's,"Polanco, Miguel Hidalgo, Ciudad de México",19.431252,-99.196062,Ciudad de México,Miguel Hidalgo,McTrío Hamb con Queso,McTrío Hamb con Queso,95.0,95.0,0.0,0.0,12,"<div class=""chakra-stack css-46p1lt"" data-qa=""..."
3,2026-04-18 19:21:23.999209,Rappi,McDonald's,"Santa Fe (Vasco de Quiroga), Cuajimalpa, Ciuda...",19.371564,-99.258652,Ciudad de México,Cuajimalpa,Coca-Cola mediana,Coca-Cola mediana,65.0,65.0,0.0,0.0,13,"<div class=""chakra-stack css-46p1lt"" data-qa=""..."
4,2026-04-18 19:21:23.999209,Rappi,McDonald's,"Santa Fe (Vasco de Quiroga), Cuajimalpa, Ciuda...",19.371564,-99.258652,Ciudad de México,Cuajimalpa,Big Mac,Big Mac,145.0,145.0,0.0,0.0,13,"<div class=""chakra-stack css-46p1lt"" data-qa=""..."


In [68]:
df.shape

(27, 16)

## 2. Valores Nulos y Limpieza

In [69]:
# Resumen de valores faltantes
print("Valores faltantes por columna:")
print(df.isnull().sum())


Valores faltantes por columna:
timestamp               0
app_name                0
store_name              0
full_address            0
lat                     0
lng                     0
city                    0
municipality            0
target_product          0
scraped_product_name    0
original_price          0
final_price             0
discount_amount         0
delivery_fee            0
eta                     0
html_snippet            0
dtype: int64


In [70]:
# Revisamos los valores nulos (productos no encontrados)
df[df.eta.isnull()]

,timestamp,app_name,store_name,full_address,lat,lng,city,municipality,target_product,scraped_product_name,original_price,final_price,discount_amount,delivery_fee,eta,html_snippet


In [71]:
# Para el EDA, nos enfocamos en registros donde se encontraron productos
df_found = df[df['scraped_product_name'] != 'Not Found'].copy()
print(f"\nRegistros con productos encontrados: {len(df_found)} de {len(df)}")


Registros con productos encontrados: 27 de 27


In [72]:
df.scraped_product_name.value_counts()

scraped_product_name
Coca-Cola mediana        9
Big Mac                  9
McTrío Hamb con Queso    9
Name: count, dtype: int64

## 3. Análisis de Precios

In [73]:
# Precios promedio por producto objetivo
price_stats = df_found.groupby('target_product')[['original_price', 'final_price']].agg(['mean', 'min', 'max'])
price_stats

original_price               final_price              
                                mean    min    max        mean    min    max
target_product                                                              
Big Mac                   129.444444  125.0  145.0  129.444444  125.0  145.0
Coca-Cola mediana          57.000000   55.0   65.0   57.000000   55.0   65.0
McTrío Hamb con Queso      95.000000   95.0   95.0   95.000000   95.0   95.0

In [74]:
fig_prices = px.box(df_found, x='target_product', y='final_price',
                    color='city', 
                    title='Distribución de Precios por Producto y Estado',
                    labels={'final_price': 'Precio Final ($)', 'target_product': 'Producto', 'city': 'Estado'})
fig_prices.show()

## 4. Métricas de Entrega

In [75]:
# Costo de envío y ETA promedio por Municipio
delivery_stats = df.groupby('municipality')[['delivery_fee', 'eta']].mean().reset_index()

fig_fee = px.bar(delivery_stats, x='municipality', y='delivery_fee', 
                 title='Costo de Envío Promedio por Municipio',
                 labels={'delivery_fee': 'Costo Promedio ($)', 'municipality': 'Municipio'})
fig_fee.show()

In [76]:
fig_eta = px.histogram(df, x='eta', title='Distribución de Tiempos de Entrega (ETA)',
                       labels={'eta': 'Minutos'})
fig_eta.show()